In [0]:
from pyspark.sql.functions import current_timestamp, lit, col, trim, when

# 1. Create Medallion Schemas
spark.sql("CREATE SCHEMA IF NOT EXISTS ai_job_lakehouse.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS ai_job_lakehouse.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS ai_job_lakehouse.gold")

# 2. Bronze Layer: Raw Ingestion + Audit Columns
raw_path = "/Volumes/ai_job_lakehouse/default/raw_landing/WA_Fn-UseC_-Telco-Customer-Churn.csv"

df_bronze = (
    spark.read.csv(raw_path, header=True, inferSchema=True)
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", lit("WA_Fn-UseC_-Telco-Customer-Churn.csv"))
)

df_bronze.write.format("delta").mode("overwrite").saveAsTable("ai_job_lakehouse.bronze.telco_customers")
print(f"Bronze layer saved! Rows: {df_bronze.count()}")

# 3. Silver Layer: Data Cleaning & Feature Engineering
df_silver = (
    df_bronze
    .dropDuplicates(["customerID"])
    # Fix blank spaces in TotalCharges for new customers (tenure = 0) and cast to double
    .withColumn(
        "TotalCharges",
        when(trim(col("TotalCharges")) == "", 0.0)
        .otherwise(col("TotalCharges").cast("double"))
    )
    # Binary target flag for ML & SQL aggregations
    .withColumn("churn_flag", when(col("Churn") == "Yes", 1).otherwise(0))
    # Business segmentation: Tenure Group
    .withColumn(
        "tenure_group",
        when(col("tenure") <= 12, "0-1 Year (New)")
        .when(col("tenure") <= 24, "1-2 Years")
        .when(col("tenure") <= 48, "2-4 Years")
        .otherwise("4+ Years (Loyal)")
    )
    # Business segmentation: Spend Tier
    .withColumn(
        "spend_tier",
        when(col("MonthlyCharges") >= 70.0, "High Spend")
        .when(col("MonthlyCharges") >= 35.0, "Medium Spend")
        .otherwise("Low Spend")
    )
)

df_silver.write.format("delta").mode("overwrite").saveAsTable("ai_job_lakehouse.silver.telco_customers_clean")
print(f"Silver layer saved! Clean rows: {df_silver.count()}")
display(df_silver.limit(5))

Bronze layer saved! Rows: 7043
Silver layer saved! Clean rows: 7043


customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,ingestion_timestamp,source_file,churn_flag,tenure_group,spend_tier
3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,2026-11-25T13:35:51.741Z,WA_Fn-UseC_-Telco-Customer-Churn.csv,1,0-1 Year (New),Medium Spend
7310-EGVHZ,Male,0,No,No,1,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Bank transfer (automatic),20.2,20.2,No,2026-11-25T13:35:51.741Z,WA_Fn-UseC_-Telco-Customer-Churn.csv,0,0-1 Year (New),Low Spend
3413-BMNZE,Male,1,No,No,1,Yes,No,DSL,No,No,No,No,No,No,Month-to-month,No,Bank transfer (automatic),45.25,45.25,No,2026-11-25T13:35:51.741Z,WA_Fn-UseC_-Telco-Customer-Churn.csv,0,0-1 Year (New),Medium Spend
6234-RAAPL,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,Yes,Yes,No,Yes,Yes,No,Two year,No,Bank transfer (automatic),99.9,7251.7,No,2026-11-25T13:35:51.741Z,WA_Fn-UseC_-Telco-Customer-Churn.csv,0,4+ Years (Loyal),High Spend
5948-UJZLF,Male,0,No,No,2,Yes,No,DSL,No,Yes,No,No,No,No,Month-to-month,No,Mailed check,49.25,97.0,No,2026-11-25T13:35:51.741Z,WA_Fn-UseC_-Telco-Customer-Churn.csv,0,0-1 Year (New),Medium Spend
